In [1]:
%pip install uv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import pickle


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load the CSV file
df = pd.read_csv(r"C:\DTT ML 2026\05-deployment\course_lead_scoring_2026.csv")

In [3]:
# Check that the dataset loaded properly and view the first few rows
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [4]:
import urllib.request
import os

base_url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/homework/05-deployment"
files = [
    "pyproject.toml",
    "uv.lock",
    "pipeline.bin",
    "model_metadata.json",
    "feature_defaults.json",
    "smoke_test.py",
    "predict.py",
    "q6_test.py",
    "Dockerfile"
]

for f in files:
    urllib.request.urlretrieve(f"{base_url}/{f}", f)
    print(f"Downloaded {f}")

Downloaded pyproject.toml
Downloaded uv.lock
Downloaded pipeline.bin
Downloaded model_metadata.json
Downloaded feature_defaults.json
Downloaded smoke_test.py
Downloaded predict.py
Downloaded q6_test.py
Downloaded Dockerfile


In [5]:
import hashlib

expected_hash = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"

with open("pipeline.bin", "rb") as f:
    computed_hash = hashlib.sha256(f.read()).hexdigest()

print(f"Computed: {computed_hash}")
print(f"Expected: {expected_hash}")
assert computed_hash == expected_hash, "Checksum mismatch! Redownload pipeline.bin."
print("Checksum verification: PASSED")

Computed: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Expected: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Checksum verification: PASSED


In [6]:
import json

with open("feature_defaults.json", "r") as f:
    defaults = json.load(f)

print("Default values used for missing inputs:")
print(json.dumps(defaults, indent=2))

Default values used for missing inputs:
{
  "number_of_courses_viewed": 2.0,
  "annual_income": 55051.0,
  "interaction_count": 4.0,
  "lead_score": 0.49
}


In [7]:
# Question 1: Check your local uv version
!uv --version

uv 0.13.5 (46776a95d 2026-10-21 x86_64-pc-windows-msvc)


In [16]:
import warnings
from sklearn.exceptions import InconsistentVersionWarning

warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

# Question 2: Locked Scikit-Learn version
with open("pyproject.toml", "r") as f:
    for line in f:
        if "scikit-learn" in line.lower():
            print("pyproject.toml:", line.strip())

with open("uv.lock", "r") as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        if 'name = "scikit-learn"' in line:
            print("uv.lock:\n" + "".join(lines[i:i+3]).strip())
            break

# Load frozen pipeline
with open("pipeline.bin", "rb") as f_in:
    pipeline = pickle.load(f_in)

def predict_lead(lead_dict):
    if hasattr(pipeline, "predict_proba"):
        return pipeline.predict_proba([lead_dict])[0, 1]
    elif isinstance(pipeline, tuple):
        dv, model = pipeline
        X = dv.transform([lead_dict])
        return model.predict_proba(X)[0, 1]
    elif isinstance(pipeline, dict):
        dv = pipeline.get("dv") or pipeline.get("dict_vectorizer")
        model = pipeline.get("model")
        X = dv.transform([lead_dict])
        return model.predict_proba(X)[0, 1]
    raise ValueError("Unrecognized pipeline format")

pyproject.toml: "scikit-learn==1.7.2",
uv.lock:
{ name = "scikit-learn" },
    { name = "uvicorn" },
]


In [9]:
lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41
}

prob_q3 = predict_lead(lead_q3)
print(f"Lead 1 Raw Probability: {prob_q3:.6f}")
print(f"Rounded (3 decimals):  {prob_q3:.3f}")

Lead 1 Raw Probability: 0.532994
Rounded (3 decimals):  0.533


In [10]:
lead_q4_q6 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
}

prob_q4_q6 = predict_lead(lead_q4_q6)
print("\n--- Questions 4 & 6 ---")
print(f"Lead 2 Raw Probability: {prob_q4_q6:.6f}")
print(f"Rounded (3 decimals):  {prob_q4_q6:.3f}")


--- Questions 4 & 6 ---
Lead 2 Raw Probability: 0.769799
Rounded (3 decimals):  0.770


In [11]:
# Inspect Dockerfile base image
with open("Dockerfile", "r") as f:
    for line in f:
        if line.strip().startswith("FROM"):
            print("Question 5 Base Image Declaration:")
            print(" ", line.strip())
            break

Question 5 Base Image Declaration:
  FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3


In [12]:
from model import normalize_record

# A partial lead missing annual_income and location
incomplete_lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "number_of_courses_viewed": 2,
    "interaction_count": 4,
    "lead_score": 0.41
}

# Automatically applies feature_defaults.json medians and 'NA' categories
clean_lead = normalize_record(incomplete_lead)
print("Sanitized Lead:", clean_lead)

prob = pipeline.predict_proba([clean_lead])[0, 1]
print(f"Safe Inferred Probability: {prob:.3f}")

Sanitized Lead: {'lead_source': 'paid_ads', 'industry': 'technology', 'employment_status': 'employed', 'location': 'NA', 'number_of_courses_viewed': 2, 'annual_income': 55051.0, 'interaction_count': 4, 'lead_score': 0.41}
Safe Inferred Probability: 0.609


In [14]:
import requests

# 1. Health check
res_health = requests.get("http://localhost:9696/health")
print("Health Status:", res_health.status_code)
print(res_health.json())

# 2. Predict Lead 2 (Question 4 & 6)
lead = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
}

res_predict = requests.post("http://localhost:9696/predict", json=lead)
print("\nPredict Status:", res_predict.status_code)
print(res_predict.json())

Health Status: 200
{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}

Predict Status: 200
{'conversion_probability': 0.769799, 'conversion': True}


In [13]:
%run q6_test.py

{'conversion_probability': 0.769799, 'conversion': True}
